<h2 style="text-align: center;"><span style="color: #3498db;">5. Ejercicio Práctico de Laboratorio</span></h2>

### 5.1 Planteamiento del problema
En esta fase final, uniremos los conceptos de búsqueda en grafos y álgebra lineal para resolver problemas de similitud y exploración de estados.

**Configuración de variables (Setup)**

Dado que estamos trabajando en un cuaderno independiente para las prácticas, primero instanciamos en memoria las dependencias, el grafo de rutas (`mapa`) y la matriz de observaciones (`dataset`) que utilizaremos en los siguientes ejercicios.

In [2]:
import numpy as np

# 1. Recrear el grafo (mapa) para el algoritmo DFS
mapa = {
    "Almacen": ["Centro", "Terminal"],
    "Centro": ["Almacen", "ZonaNorte", "Mercado"],
    "Terminal": ["Almacen", "Mercado"],
    "Mercado": ["Centro", "Terminal", "ZonaNorte"],
    "ZonaNorte": ["Centro", "Mercado"],
}

# 2. Recrear el dataset bidimensional para el cálculo de distancias
dataset = np.array([
    [22.5, 60.], 
    [19.0, 45.], 
    [25.3, 70.], 
    [21.0, 50.]
])

print("Entorno de datos (mapa y dataset) cargado correctamente para la práctica.")

Entorno de datos (mapa y dataset) cargado correctamente para la práctica.


### 5.2 Parte A: Búsqueda en Profundidad (DFS)
A diferencia de la exploración por niveles, la Búsqueda en Profundidad (DFS) explora una rama del grafo hasta su límite antes de retroceder (*backtracking*).

#### Análisis Teórico: BFS vs DFS

| Característica | Búsqueda en Amplitud (BFS) | Búsqueda en Profundidad (DFS) |
| :--- | :--- | :--- |
| **Estructura de Datos** | Cola (FIFO: Primero en entrar, primero en salir) | Pila (LIFO: Último en entrar, primero en salir) |
| **Estrategia de exploración** | Expansión por niveles (anillos concéntricos). | Inmersión profunda por una sola rama hasta el final. |
| **Garantía de optimización** | **Sí.** Encuentra siempre el camino con menos saltos. | **No.** Encuentra el primer camino disponible (subóptimo). |

In [3]:
def dfs(grafo, inicio, objetivo):
    """
    Busca un camino entre 'inicio' y 'objetivo' usando DFS.
    Utiliza una Pila (LIFO).
    """
    pila = [[inicio]]
    visitados = set()
    
    while pila:
        # LIFO: Extraemos el ÚLTIMO elemento insertado en la pila
        camino = pila.pop() 
        nodo_actual = camino[-1]
        
        if nodo_actual == objetivo:
            return camino
            
        if nodo_actual not in visitados:
            visitados.add(nodo_actual)
            
            for vecino in grafo[nodo_actual]:
                nuevo_camino = camino + [vecino]
                pila.append(nuevo_camino)
                
    return None

resultado_dfs = dfs(mapa, "Almacen", "ZonaNorte")
print("Camino encontrado con DFS:", resultado_dfs)

Camino encontrado con DFS: ['Almacen', 'Terminal', 'Mercado', 'ZonaNorte']


### Respuesta a la Guía: Comparación BFS vs DFS

**1. ¿Son el mismo camino?**
No, los algoritmos encontraron rutas diferentes hacia el mismo destino ("ZonaNorte") partiendo desde "Almacén":
* **Camino encontrado por BFS:** `['Almacen', 'Centro', 'ZonaNorte']`
* **Camino encontrado por DFS:** `['Almacen', 'Terminal', 'Mercado', 'ZonaNorte']`

**2. ¿Cuál tiene menos pasos?**
El algoritmo BFS tiene menos pasos. 
* **BFS** llegó al objetivo en solo 2 pasos (saltos entre nodos).
* **DFS** tomó 3 pasos. 

**Conclusión:** Esto demuestra el principio teórico de que BFS *siempre garantiza encontrar el camino más corto* en grafos sin pesos al explorar por niveles. Por su parte, DFS simplemente devuelve la primera ruta que encuentra al sumergirse por una rama, la cual rara vez es la óptima.

### 5.3 Encontrar la observación más cercana (Similitud Vectorial)

Para que un modelo de Inteligencia Artificial (como *K-Nearest Neighbors*) pueda clasificar un dato nuevo, necesita medir matemáticamente qué tan cerca está de las experiencias pasadas.

**Objetivo:** Evaluar un nuevo registro de sensor `[20.5 °C, 48.0 % de Humedad]` y buscar su "vecino más cercano" dentro del `dataset` histórico.

| Fases del Algoritmo | Operación Matemática | Equivalencia en NumPy |
| :--- | :--- | :--- |
| **1. Iteración** | Extraer vector histórico $\vec{x}_i$ | `for obs in dataset:` |
| **2. Diferencia vectorial** | $\vec{x}_{nuevo} - \vec{x}_i$ | `nueva_observacion - obs` |
| **3. Distancia Euclidiana** | $d = \sqrt{\sum (\Delta x)^2}$ | `np.linalg.norm(...)` |
| **4. Minimización** | $\arg\min (d)$ | `np.argmin(distancias)` |

In [5]:
# 1. Definir la nueva observación (Temperatura, Humedad)
nueva_observacion = np.array([20.5, 48.0])
distancias = []

print("--- Evaluación de Similitud Vectorial (Procedimiento Detallado) ---")
print(f"Buscando similitud para el sensor nuevo: [Temp {nueva_observacion[0]}°C, Hum {nueva_observacion[1]}%]")
print("Fórmula Euclidiana: d = √((T_nuevo - T_obs)² + (H_nuevo - H_obs)²)\n")

# 2. Iterar sobre cada fila del dataset desglosando la matemática
for i, obs in enumerate(dataset):
    # Paso A: Diferencias exactas entre coordenadas
    delta_temp = nueva_observacion[0] - obs[0]
    delta_hum = nueva_observacion[1] - obs[1]
    
    # Paso B: Elevar al cuadrado cada diferencia
    cuadrado_temp = delta_temp ** 2
    cuadrado_hum = delta_hum ** 2
    
    # Paso C: Sumar los cuadrados y sacar la raíz (Distancia Euclidiana)
    suma_cuadrados = cuadrado_temp + cuadrado_hum
    distancia = np.sqrt(suma_cuadrados)
    distancias.append(distancia)
    
    # Imprimir el procedimiento paso a paso para la consola
    print(f"Evaluando Sensor Histórico {i} {obs}:")
    print(f"  1. Restas       -> ΔTemp: ({nueva_observacion[0]} - {obs[0]}) = {delta_temp:.2f}  |  ΔHum: ({nueva_observacion[1]} - {obs[1]}) = {delta_hum:.2f}")
    print(f"  2. Cuadrados    -> ({delta_temp:.2f})² = {cuadrado_temp:.2f}  |  ({delta_hum:.2f})² = {cuadrado_hum:.2f}")
    print(f"  3. Suma y Raíz  -> √({cuadrado_temp:.2f} + {cuadrado_hum:.2f}) = √({suma_cuadrados:.2f}) = {distancia:.4f}\n")

# 3. Identificar el índice con la menor distancia (el más cercano)
indice_minimo = np.argmin(distancias)
obs_mas_cercana = dataset[indice_minimo]

print("-" * 65)
print(f"RESULTADO: El registro histórico más cercano es el Índice {indice_minimo} {obs_mas_cercana}")

--- Evaluación de Similitud Vectorial (Procedimiento Detallado) ---
Buscando similitud para el sensor nuevo: [Temp 20.5°C, Hum 48.0%]
Fórmula Euclidiana: d = √((T_nuevo - T_obs)² + (H_nuevo - H_obs)²)

Evaluando Sensor Histórico 0 [22.5 60. ]:
  1. Restas       -> ΔTemp: (20.5 - 22.5) = -2.00  |  ΔHum: (48.0 - 60.0) = -12.00
  2. Cuadrados    -> (-2.00)² = 4.00  |  (-12.00)² = 144.00
  3. Suma y Raíz  -> √(4.00 + 144.00) = √(148.00) = 12.1655

Evaluando Sensor Histórico 1 [19. 45.]:
  1. Restas       -> ΔTemp: (20.5 - 19.0) = 1.50  |  ΔHum: (48.0 - 45.0) = 3.00
  2. Cuadrados    -> (1.50)² = 2.25  |  (3.00)² = 9.00
  3. Suma y Raíz  -> √(2.25 + 9.00) = √(11.25) = 3.3541

Evaluando Sensor Histórico 2 [25.3 70. ]:
  1. Restas       -> ΔTemp: (20.5 - 25.3) = -4.80  |  ΔHum: (48.0 - 70.0) = -22.00
  2. Cuadrados    -> (-4.80)² = 23.04  |  (-22.00)² = 484.00
  3. Suma y Raíz  -> √(23.04 + 484.00) = √(507.04) = 22.5175

Evaluando Sensor Histórico 3 [21. 50.]:
  1. Restas       -> ΔTemp: (20.

### Interpretación de los Resultados (Similitud Vectorial)

Tras ejecutar el cálculo de la distancia euclidiana desglosada, podemos extraer las siguientes conclusiones sobre nuestro nuevo registro (`[20.5 °C, 48.0 %]`):

| Sensor Evaluado | Distancia ($\approx$) | Análisis de Similitud |
| :--- | :---: | :--- |
| **Índice 0** `[22.5, 60.0]` | 12.16 | **Baja similitud.** Hay una brecha considerable, especialmente en la humedad (12% de diferencia). |
| **Índice 1** `[19.0, 45.0]` | 3.35 | **Alta similitud.** Es el segundo registro más cercano, con variaciones pequeñas en ambas variables. |
| **Índice 2** `[25.3, 70.0]` | 22.51 | **Nula similitud.** Es el sensor más alejado de todos, representando un ambiente mucho más cálido y húmedo. |
| **Índice 3** `[21.0, 50.0]` | **2.06** | **Similitud Máxima (Vecino más cercano).** La variación es mínima: solo 0.5 °C en temperatura y 2.0 % en humedad. |

**Conclusión Práctica:**
Si este algoritmo fuera la base de un sistema predictivo (como K-Nearest Neighbors), la IA clasificaría el nuevo sensor basándose en el historial del **Índice 3**. Esto demuestra cómo el álgebra lineal permite a la máquina "entender" qué tan parecidas son dos situaciones del mundo real, convirtiendo características físicas en coordenadas geométricas.